# Results analysis

This notebook summarizes `scores_comparison.csv` from the main experimental run and writes the paper tables under `notebooks/tables/`.

**Input:** `runs/final_run/scores_comparison.csv` — macro F1 and problematic-generation rate for each (dataset, model, prompting, definition) setting.

**Outputs:**
- `tables/scores_summary.csv` — mean ± std macro F1 by model, definition, and prompting
- `tables/scores.csv` — wide macro-F1 table (prompting × definition, with dataset × model columns)
- `tables/problematic_generation_rates.csv` — settings with a non-zero parse-failure rate
- `tables/mean_confusion_matrix_few-shot_no_definition.csv` — mean ± std confusion matrix over HateCheck, no-definition, few-shot nearest
- `tables/mean_confusion_matrix.csv` — mean ± std confusion matrix over the `best_run` HateCheck glob used in the paper


## Load and filter scores

Keep the three evaluation datasets used in the paper (HateCheck, Reddit-relabeled, Bulgaria-relabeled) and the two definition conditions (`no_definition`, `hsc`). Drop `vanilla` and `target_excluded` variants. Strip the `-no-reasoning` suffix from prompting names if present.


In [49]:
from pathlib import Path

import glob
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
RUNS_DIR = PROJECT_ROOT / "runs"
TABLES_DIR = Path("tables")
TABLES_DIR.mkdir(exist_ok=True)

SCORES_CSV = RUNS_DIR / "final_run" / "scores_comparison.csv"
DATASETS = ["hatecheck", "hatecheck_bulgaria", "hatecheck_reddit"]
MODEL_ORDER = ["gemma2", "llama3", "qwen2"]
PROMPTING_ORDER = [
    "zero-shot",
    "few-shot-random",
    "few-shot-diverse",
    "few-shot-nearest-query",
]
DEFINITION_PREFIX = {
    "hatecheck": "hatecheck_",
    "hatecheck_bulgaria": "bulgaria_",
    "hatecheck_reddit": "reddit_",
}


def model_base(names: pd.Series) -> pd.Series:
    """Map checkpoint ids (e.g. gemma2_2b_it) to the short model family name."""
    return names.str.split("_").str[0]


def definition_scope(row: pd.Series) -> str:
    """Strip the dataset prefix so labels are `no_definition` or `hsc`."""
    return row["definition"].replace(DEFINITION_PREFIX[row["dataset"]], "", 1)


scores_df = pd.read_csv(SCORES_CSV)
scores_df = scores_df[
    scores_df["dataset"].isin(DATASETS)
    & scores_df["definition"].str.contains("no_definition|hsc")
    & ~scores_df["definition"].str.contains("vanilla|target_excluded")
].copy()
scores_df["prompting"] = scores_df["prompting"].str.replace("-no-reasoning", "", regex=False)
scores_df["model_base"] = model_base(scores_df["model"])
scores_df["definition_scope"] = scores_df.apply(definition_scope, axis=1)

print(f"Loaded {len(scores_df)} settings from {SCORES_CSV.name}")
scores_df.head()

Loaded 72 settings from scores_comparison.csv


,dataset,model,prompting,definition,macro_f1_score,percentage_of_problematic_generations,model_base,definition_scope
0,hatecheck,gemma2_2b_it,zero-shot,hatecheck_no_definition,0.741,0.0,gemma2,no_definition
1,hatecheck,gemma2_2b_it,zero-shot,hatecheck_hsc,0.845,0.0,gemma2,hsc
2,hatecheck,gemma2_2b_it,few-shot-random,hatecheck_no_definition,0.789,0.0,gemma2,no_definition
3,hatecheck,gemma2_2b_it,few-shot-random,hatecheck_hsc,0.787,0.0,gemma2,hsc
4,hatecheck,gemma2_2b_it,few-shot-diverse,hatecheck_no_definition,0.855,0.0,gemma2,no_definition


## Macro F1 by experimental factor

Mean and standard deviation of macro F1, pooling over the other factors. Model names are shortened to the family prefix.


In [50]:
def mean_std_f1(df: pd.DataFrame, by: str, *, shorten_model: bool = False) -> pd.DataFrame:
    """Return mean and std of macro_f1_score grouped by `by`, with MultiIndex columns."""
    grouped = df.groupby(by)["macro_f1_score"].agg(["mean", "std"]).round(3)
    grouped.columns = pd.MultiIndex.from_product([["macro_f1_score"], grouped.columns])
    if shorten_model:
        grouped.index = model_base(grouped.index.to_series())
        grouped.index.name = "model"
    return grouped


def stack_factor_summaries(summaries: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Stack per-factor tables into one CSV-friendly frame with an `aspect` index."""
    frames = []
    for aspect, table in summaries.items():
        frame = table.reset_index().rename(columns={table.index.name: "value"})
        frame.insert(0, "aspect", aspect)
        frames.append(frame)
    return pd.concat(frames, ignore_index=True).set_index("aspect")


factor_summaries = {
    "model": mean_std_f1(scores_df, "model", shorten_model=True),
    "definition": mean_std_f1(scores_df, "definition"),
    "prompting": mean_std_f1(scores_df, "prompting"),
}

for aspect, table in factor_summaries.items():
    print(aspect)
    display(table)

scores_summary = stack_factor_summaries(factor_summaries)
scores_summary.to_csv(TABLES_DIR / "scores_summary.csv")
scores_summary

model


macro_f1_score       
                 mean    std
model                       
gemma2          0.696  0.118
llama3          0.734  0.127
qwen2           0.740  0.083

definition


macro_f1_score       
                                  mean    std
definition                                   
bulgaria_hsc                     0.602  0.071
bulgaria_no_definition           0.602  0.091
hatecheck_hsc                    0.828  0.048
hatecheck_no_definition          0.821  0.071
reddit_hsc                       0.750  0.038
reddit_no_definition             0.736  0.051

prompting


macro_f1_score       
                                 mean    std
prompting                                   
few-shot-diverse                0.732  0.095
few-shot-nearest-query          0.772  0.107
few-shot-random                 0.715  0.098
zero-shot                       0.674  0.128

value macro_f1_score       
                                              mean    std
aspect                                                   
model                        gemma2          0.696  0.118
model                        llama3          0.734  0.127
model                         qwen2          0.740  0.083
definition             bulgaria_hsc          0.602  0.071
definition   bulgaria_no_definition          0.602  0.091
definition            hatecheck_hsc          0.828  0.048
definition  hatecheck_no_definition          0.821  0.071
definition               reddit_hsc          0.750  0.038
definition     reddit_no_definition          0.736  0.051
prompting          few-shot-diverse          0.732  0.095
prompting    few-shot-nearest-query          0.772  0.107
prompting           few-shot-random          0.715  0.098
prompting                 zero-shot          0.674  0.128

## Best setting per model

For each model, the row with the highest macro F1 among the filtered settings.


In [51]:
best_idx = scores_df.groupby("model")["macro_f1_score"].idxmax()
best_setting_df = scores_df.loc[best_idx].reset_index(drop=True)
best_setting_df["model"] = model_base(best_setting_df["model"])
best_setting_df

,dataset,model,prompting,definition,macro_f1_score,percentage_of_problematic_generations,model_base,definition_scope
0,hatecheck,gemma2,few-shot-diverse,hatecheck_no_definition,0.855,0.0,gemma2,no_definition
1,hatecheck,llama3,few-shot-nearest-query,hatecheck_no_definition,0.929,0.0,llama3,no_definition
2,hatecheck,qwen2,few-shot-nearest-query,hatecheck_hsc,0.898,0.0,qwen2,hsc


## Wide macro-F1 table

One row per (prompting, definition scope); columns are dataset × model. Written to `tables/scores.csv`.


In [55]:
wide_scores = scores_df.pivot_table(
    index=["prompting", "definition_scope"],
    columns=["dataset", "model_base"],
    values="macro_f1_score",
    aggfunc="first",
)
wide_scores = wide_scores.reindex(PROMPTING_ORDER, level=0)
wide_scores = wide_scores.reindex(["no_definition", "hsc"], level=1)
wide_scores = wide_scores.reindex(
    columns=pd.MultiIndex.from_product(
        [DATASETS, MODEL_ORDER], names=["dataset", "model"]
    )
)
wide_scores.index.names = ["prompting", "definition"]
wide_scores.to_csv(TABLES_DIR / "scores.csv")
wide_scores

dataset                              hatecheck                \
model                                   gemma2 llama3  qwen2   
prompting              definition                              
zero-shot              no_definition     0.741  0.796  0.835   
                       hsc               0.845  0.780  0.805   
few-shot-random        no_definition     0.789  0.854  0.766   
                       hsc               0.787  0.834  0.804   
few-shot-diverse       no_definition     0.855  0.875  0.675   
                       hsc               0.848  0.853  0.756   
few-shot-nearest-query no_definition     0.841  0.929  0.896   
                       hsc               0.810  0.920  0.898   

dataset                              hatecheck_bulgaria                \
model                                            gemma2 llama3  qwen2   
prompting              definition                                       
zero-shot              no_definition              0.442  0.487  0.546   
                       hsc                        0.543  0.496  0.597   
few-shot-random        no_definition              0.552  0.586  0.692   
                       hsc                        0.560  0.541  0.659   
few-shot-diverse       no_definition              0.579  0.620  0.737   
                       hsc                        0.577  0.593  0.734   
few-shot-nearest-query no_definition              0.592  0.697  0.700   
                       hsc                        0.570  0.702  0.658   

dataset                              hatecheck_reddit                
model                                          gemma2 llama3  qwen2  
prompting              definition                                    
zero-shot              no_definition            0.628  0.681  0.736  
                       hsc                      0.733  0.680  0.758  
few-shot-random        no_definition            0.728  0.772  0.716  
                       hsc                      0.722  0.743  0.770  
few-shot-diverse       no_definition            0.752  0.783  0.696  
                       hsc                      0.751  0.755  0.740  
few-shot-nearest-query no_definition            0.746  0.806  0.787  
                       hsc                      0.717  0.825  0.800

## Problematic generation rates

Settings where at least one model output failed to parse. Rates are fractions of the test set.


In [56]:
problematic_df = (
    scores_df[scores_df["percentage_of_problematic_generations"] > 0]
    .sort_values("percentage_of_problematic_generations", ascending=False)
    .drop(columns=["macro_f1_score", "model_base", "definition_scope"])
)
problematic_df.to_csv(TABLES_DIR / "problematic_generation_rates.csv", index=False)
problematic_df

,dataset,model,prompting,definition,percentage_of_problematic_generations
39,hatecheck_bulgaria,llama3_2_3b_instruct,few-shot-random,bulgaria_no_definition,0.003
32,hatecheck,llama3_2_3b_instruct,few-shot-diverse,hatecheck_no_definition,0.002
42,hatecheck_bulgaria,llama3_2_3b_instruct,few-shot-diverse,bulgaria_no_definition,0.002
52,hatecheck_reddit,llama3_2_3b_instruct,few-shot-diverse,reddit_no_definition,0.002
28,hatecheck,llama3_2_3b_instruct,zero-shot,hatecheck_no_definition,0.001
30,hatecheck,llama3_2_3b_instruct,few-shot-random,hatecheck_no_definition,0.001
43,hatecheck_bulgaria,llama3_2_3b_instruct,few-shot-diverse,bulgaria_hsc,0.001
50,hatecheck_reddit,llama3_2_3b_instruct,few-shot-random,reddit_no_definition,0.001


## Mean confusion matrices

Each `confusion_matrix.txt` is a 2×2 sklearn table (rows = gold, columns = predicted). We add margins, then report mean ± std over the matched run directories. Std uses the population formula (divide by *n*) to match the original paper tables.


In [58]:
CM_INDEX = ["non-hateful-actual", "hateful-actual", "total-actual"]
CM_COLUMNS = ["non-hateful-predicted", "hateful-predicted", "total-predicted"]


def read_confusion_matrix(path: Path) -> pd.DataFrame:
    """Parse a 2×2 confusion_matrix.txt and append row/column totals."""
    lines = Path(path).read_text().splitlines()
    nn, nh = map(int, lines[4].split()[1:])
    hn, hh = map(int, lines[5].split()[1:])
    return pd.DataFrame(
        [
            [nn, nh, nn + nh],
            [hn, hh, hn + hh],
            [nn + hn, nh + hh, nn + nh + hn + hh],
        ],
        index=CM_INDEX,
        columns=CM_COLUMNS,
    )


def mean_std_confusion_matrices(paths) -> pd.DataFrame:
    """Format mean ± std of confusion matrices (with margins) over `paths`."""
    paths = list(paths)
    if not paths:
        raise FileNotFoundError("No confusion_matrix.txt files matched the glob.")
    stacked = pd.concat([read_confusion_matrix(p) for p in paths])
    mean = stacked.groupby(level=0).mean().reindex(CM_INDEX)[CM_COLUMNS]
    std = stacked.groupby(level=0).std(ddof=0).reindex(CM_INDEX)[CM_COLUMNS]
    return mean.round(1).astype(str) + " ± " + std.round(1).astype(str)


def save_mean_confusion_matrix(glob_pattern: str, output_name: str) -> pd.DataFrame:
    paths = sorted(glob.glob(glob_pattern))
    print(f"{output_name}: {len(paths)} matrices")
    table = mean_std_confusion_matrices(paths)
    table.to_csv(TABLES_DIR / output_name)
    display(table)
    return table


save_mean_confusion_matrix(
    str(
        RUNS_DIR
        / "final_run"
        / "hatecheck"
        / "*"
        / "hatecheck_no_definition"
        / "few-shot-nearest-query"
        / "confusion_matrix.txt"
    ),
    "mean_confusion_matrix_few-shot_no_definition.csv",
)

save_mean_confusion_matrix(
    str(RUNS_DIR / "best_run" / "hatecheck" / "*" / "*" / "*" / "confusion_matrix.txt"),
    "mean_confusion_matrix.csv",
)
pass

mean_confusion_matrix_few-shot_no_definition.csv: 3 matrices


,non-hateful-predicted,hateful-predicted,total-predicted
non-hateful-actual,1130.3 ± 175.6,226.7 ± 175.6,1357.0 ± 0.0
hateful-actual,139.7 ± 98.4,2137.0 ± 98.4,2276.7 ± 0.5
total-actual,1270.0 ± 265.7,2363.7 ± 265.9,3633.7 ± 0.5


mean_confusion_matrix.csv: 36 matrices


,non-hateful-predicted,hateful-predicted,total-predicted
non-hateful-actual,983.3 ± 189.1,373.6 ± 189.1,1356.9 ± 0.2
hateful-actual,204.0 ± 277.3,2072.6 ± 277.1,2276.6 ± 1.1
total-actual,1187.4 ± 432.3,2446.2 ± 432.2,3633.5 ± 1.3
